# Chapter 8: Foundational models and Semantic IDs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kimfalk/modern-recommender-systems/blob/main/notebooks/chapter-06/example_notebook.ipynb)


In [ ]:
# Install datasets library for HuggingFace
!pip install -q datasets
!pip install -q mlflow  

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "recsys").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Added to sys.path: {project_root}")

In [ ]:
from recsys.utils.colab import setup_colab_environment, get_data_path, check_gpu
# One-line setup for Colab users
setup_colab_environment()

# Check GPU availability
check_gpu()

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from recsys.data import loaders
from recsys.semantic_ids.utils import prepare_data
from recsys.semantic_ids.training import run_pipeline_debug
from recsys.semantic_ids.evaluations import evaluate_semantic_ids

DATA_PATH = get_data_path()

In [ ]:
print("Ready to build recommender systems!")

In [ ]:
from recsys.data.loaders import load_movielens_descriptions

movies = load_movielens_descriptions(data_dir=DATA_PATH)

movies.head()

In [ ]:
# Reload the loaders module to pick up latest changes
import importlib
import recsys.data.loaders
importlib.reload(recsys.data.loaders)

In [ ]:
# Load movie descriptions from HuggingFace
from recsys.data.loaders import load_movielens_descriptions

movies_descriptions = load_movielens_descriptions(data_dir=DATA_PATH)

In [ ]:
# Check the structure and columns

movies_descriptions.rename(columns={'Overview': 'description',
                                    'Title': 'title',
                                    'Genre': 'genres'}, inplace=True)
print(f"Shape: {movies_descriptions.shape}")
print(f"\nColumns: {movies_descriptions.columns.tolist()}")
movies_descriptions.head()

In [ ]:
movies_descriptions = movies_descriptions[~movies_descriptions['title'].isna()]
movies_descriptions.shape



In [ ]:
codebook_sizes = [16, 32, 128]  # Tiny capacity: max 512 combinations for 9,837 movies
internal_dim = 512  # Match input dimension
usage_loss_weight = 2.0  # Very strong penalty for imbalanced usage
epochs = 500  # Fewer epochs for faster iteration

In [ ]:
import mlflow
import mlflow.pytorch

# Set experiment name
mlflow.set_experiment("semantic-ids-experiments")


In [ ]:
recreate_embeddings = True

In [ ]:
# Reload the module to pick up the fixes
import importlib
import recsys.semantic_ids.semantic_ids_pipeline
importlib.reload(recsys.semantic_ids.semantic_ids_pipeline)
importlib.reload(recsys.semantic_ids.training)
importlib.reload(recsys.semantic_ids.rqvae)
importlib.reload(recsys.semantic_ids.vector_quantizer)
importlib.reload(recsys.semantic_ids.evaluations)

# Now test with the fixed code
from recsys.semantic_ids.semantic_ids_pipeline import SemanticIDPipeline

from recsys.semantic_ids.utils import prepare_data

with mlflow.start_run(run_name="aggressive-clustering"):
    
    # Log parameters
    mlflow.log_params({
        "codebook_sizes": codebook_sizes,
        "internal_dim": internal_dim,
        "usage_loss_weight": usage_loss_weight,
        "epochs": epochs,
        "variance_weight": 1.0  # if using variance regularization
    })
   
    pipeline = SemanticIDPipeline(codebook_sizes, 
                                internal_dim,
                                usage_loss_weight=usage_loss_weight)

    if recreate_embeddings:
        texts = prepare_data(movies_descriptions)
        print("\nEncoding texts...")
        embeddings = pipeline.text_encoder.encode(texts, show_progress_bar=True)
        recreate_embeddings = False
            
    data_tensor = pipeline.initialize_data_with_embeddings(embeddings)
    pipeline.train(data_tensor, epochs=epochs)

    df_enriched = pipeline.inference(movies_descriptions, data_tensor)
    evaluate_semantic_ids(pipeline, df_enriched, data_tensor, codebook_sizes)
    # Log evaluation metrics
    num_clusters = df_enriched['semantic_id'].nunique()
    avg_cluster_size = len(df_enriched) / num_clusters
    
    mlflow.log_metrics({
        "num_unique_clusters": num_clusters,
        "avg_cluster_size": avg_cluster_size,
        "total_items": len(df_enriched)
    })
    
    # Save and log the model
    import torch
    import os
    
    # Create artifacts directory if it doesn't exist
    os.makedirs("mlflow_artifacts", exist_ok=True)
    
    # Save model state dict
    model_path = "mlflow_artifacts/rqvae_model.pth"
    torch.save(pipeline.rqvae.state_dict(), model_path)
    
    # Log as artifact
    mlflow.log_artifact(model_path)

In [ ]:
# Enhanced diagnostics - check loss components
import torch

print("="*70)
print("TRAINING DIAGNOSTICS")
print("="*70)

# 1. Check encoder output variance
with torch.no_grad():
    encoder_output = pipeline.rqvae.encoder(data_tensor[:100])
    print(f"\n1. Encoder Output:")
    print(f"   Std: {encoder_output.std().item():.4f}")
    print(f"   Mean: {encoder_output.mean().item():.4f}")
    print(f"   Min: {encoder_output.min().item():.4f}")
    print(f"   Max: {encoder_output.max().item():.4f}")

# 2. Check codebook usage and get actual loss values
print(f"\n2. Codebook Usage:")
for level_idx, vq in enumerate(pipeline.rqvae.quantizers):
    with torch.no_grad():
        if level_idx == 0:
            z = pipeline.rqvae.encoder(data_tensor)
        else:
            z = pipeline.rqvae.encoder(data_tensor)
            for prev_vq in pipeline.rqvae.quantizers[:level_idx]:
                z, _, _ = prev_vq(z)
        
        # Get loss components
        vq.train()
        quantized, vq_loss, indices = vq(z)
        
        unique_codes = torch.unique(indices).numel()
        
    print(f"\n   Level {level_idx+1} (size={codebook_sizes[level_idx]}):")
    print(f"   - Unique codes: {unique_codes}/{codebook_sizes[level_idx]} ({100*unique_codes/codebook_sizes[level_idx]:.1f}%)")
    print(f"   - VQ loss: {vq_loss.item():.4f}")
    print(f"   - Usage weight: {vq.usage_loss_weight}")

# 3. Check full forward pass
print(f"\n3. Full Forward Pass:")
with torch.no_grad():
    reconstructed, total_loss, codes = pipeline.rqvae(data_tensor[:100])
    recon_error = ((data_tensor[:100] - reconstructed) ** 2).mean()
    
    print(f"   - Reconstruction error: {recon_error.item():.6f}")
    print(f"   - Total VQ loss: {total_loss.item():.4f}")

print("\n" + "="*70)


In [ ]:
df_enriched.head()


In [ ]:

importlib.reload(recsys.semantic_ids.evaluations)
from matplotlib.pyplot import title
from recsys.semantic_ids.evaluations import test_semantic_coherence

# test_semantic_coherence(df_enriched)
def get_row(title, df):
    return df[df['title'] == title]['semantic_id'].values[0]
get_row("Star Wars", df_enriched)


In [ ]:
df_enriched['title'].str.startswith('Star Wars')


In [ ]:
df_enriched = df_enriched[df_enriched.isna() == False]

df_enriched[df_enriched['title'].isna()]



In [ ]:
df_enriched[df_enriched['title'].str.startswith('Star Wars')]


In [ ]:

def investigate_cluster(df):# Check what's in the largest cluster
    largest_cluster_id = df['semantic_id'].value_counts().head(1).index[0]
    largest_cluster = df[df['semantic_id'] == largest_cluster_id]

    print(f"Items in cluster {largest_cluster_id}: {len(largest_cluster)}")
    print("\nSample titles:")
    print(largest_cluster[['title', 'genres']].head(20))

    print("\nGenre distribution:")
    
    print(largest_cluster['genres'].value_counts().head(10))

investigate_cluster(df_enriched)
